# La búsqueda del techo (v1)

Antes de comparar nada hay que elegir un escalar: hasta dónde sube la rampa del
término de adaptación de cada familia. Ese escalar lo eligen **mirando
resultados**, así que la cosa que lo elige es un experimento y necesita todo lo
que un experimento necesita — su propia escala, su propio rol del material, su
propia regla de desempate — y hasta ahora no tenía informe propio. Sus seis
celdas vivían adentro del reporte de campaña, que es el lugar donde se leen sus
consecuencias y no donde se juzga cómo se llegó a ellas.

Este cuaderno es ese informe. También corre el **ensayo local**, que es una cosa
distinta de la búsqueda y conviene no confundir:

| | escala | dónde escribe | su respuesta |
| --- | --- | --- | --- |
| **búsqueda** | 20 épocas · 3 semillas | `ceilings.json` | rige la campaña |
| **ensayo** | pocas épocas · 1 semilla | `ceilings.pilot.json` | **no se cita** |

El ensayo contesta *«¿el programa corre?»*, que es lo único que un ensayo puede
contestar. No contesta *«¿cuál es el techo?»*: la rampa avanza con la fracción de
entrenamiento transcurrida, así que a escala corta se satura en la segunda época
y todo techo se alcanza casi enseguida. Lo que mediría es un paisaje donde nada
más entrena.

Por eso son dos archivos y no uno. Con uno solo, el ensayo habría escrito donde
va la respuesta que la campaña consume, y una corrida completa lo habría gastado
sin una palabra.

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness, tables


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo."""
    display(Markdown(text))

## 1 · Qué registro rige ahora mismo

Leído del disco, no recordado. El registro completo le gana siempre al ensayo:
un ensayo no desplaza una medición.

In [ ]:
proc = config.ceilings_provenance()
lineas = [f"**registro en vigor:** `{proc['source']}`"]
if proc["record"]:
    lineas.append(f"`{Path(proc['record']).name}` · "
                  f"{proc['epochs']} épocas · {proc['seeds']} semilla(s)")
    if proc.get("atRequiredScale") is False:
        lineas.append("**Está por debajo de la escala que el protocolo declara "
                      f"({proc['requiredScale']}): su techo no se cita.**")
else:
    lineas.append("ninguno todavía — ni la búsqueda ni el ensayo han corrido.")
show(" · ".join(lineas))

## 2 · El ensayo local

Corre el mismo programa que la búsqueda, a su propia escala declarada, y escribe
a su propio archivo. Si ya existe un registro —el que sea— no vuelve a buscar: un
registro que existe significa que la búsqueda contestó, y volver a contestarla
porque un llamador posterior quería otra respuesta es exactamente el silencio que
la negativa de la campaña existe para impedir.

Correrlo no es cosa de este cuaderno: lo corre `Benchmark_Search_Pilot_v1.ipynb`,
que es el cuaderno del ensayo, y a ése lo ejecuta el paso `search-pilot`. Acá se
declara lo que cuesta y se lee lo que dejó.

In [ ]:
aforo = config.search_sizing()
bajo, alto = config.CEILING_RANGE
show(
    f"Motor: **{aforo['engine']}** con `GPSampler`.  \n"
    f"La búsqueda real: **{aforo['trials']} trials** por "
    f"`(familia, transferencia)` de **{aforo['epochs']} épocas** — "
    f"{aforo['families']} familias × {aforo['transfers']} transferencias × "
    f"{aforo['trials']} = **{aforo['runs']} corridas**.  \n"
    f"El ensayo: **{config.PILOT_SEARCH_TRIALS} trials** de "
    f"**{config.PILOT_SEARCH_EPOCHS} épocas**, que es "
    f"{config.PILOT_SEARCH_TRIALS * aforo['families'] * aforo['transfers']} "
    f"corridas.  \n"
    f"El rango es continuo, `[{bajo:g}, {alto:g}]` en escala logarítmica, y la "
    f"meseta la define la resolución del criterio sobre el rol de búsqueda: "
    f"**{config.SEARCH_RESOLUTION:g}**, que es una bolsa de las "
    f"{config.VALID_BAGS}."
)

In [ ]:
# La búsqueda no se corre desde acá, ni comentada. Esta celda decía
# `# harness.run_search(pilot=True)` y existía porque era la única forma de pedir
# el ensayo desde un cuaderno; hoy lo corre `Benchmark_Search_Pilot_v1.ipynb`, que
# es raíz declarada de `search-pilot`. Descomentarla habría dado dos dueños al
# ensayo de la búsqueda y habría hecho que abrir este informe cueste lo que cuesta
# correrla.

## 3 · Lo que la búsqueda eligió

Del registro en vigor. Si es el del ensayo, todo lo de abajo es plumbing y está
dicho arriba.

In [ ]:
show(tables.objective("ceilings"))

In [ ]:
# Una sola forma para toda la clase: la omisión YA significa «el que rige», así
# que el repliegue escrito a mano acá era la segunda ortografía de la misma
# regla --- y la que el informe no tenía.
registro = harness.search_record()
show(harness.search_source_note())
show(tables.render_ceilings(registro, markdown=True))

In [ ]:
show(tables.conclusion_ceilings(registro))

### 3b · Qué techo rige en cada transferencia

La búsqueda mide unas pocas transferencias y las demás heredan. Esa herencia es
una aplicación **fuera de muestra** y se declara como tal.

In [ ]:
show(tables.objective("ceilings.byTransfer"))

In [ ]:
transferencias = [f"{a}->{b}" for a, b in config.VERDICT_TRANSFERS]
show(tables.render_ceilings_by_transfer(registro, transferencias, markdown=True))

In [ ]:
show(tables.conclusion_ceilings_by_transfer(registro, transferencias))

## 4 · Lo que esta búsqueda **no** midió

Es la limitación más grande del escalar que gobierna toda la campaña, y no se lee
en ninguna otra parte. La búsqueda midió unas transferencias y las otras heredaron
su techo sin que nadie lo comprobara ahí.

In [ ]:
medidas = {f"{a}->{b}" for a, b in config.SEARCH_TRANSFERS}
todas = [f"{a}->{b}" for a, b in config.VERDICT_TRANSFERS]
heredadas = [t for t in todas if t not in medidas]
show(f"**Medidas:** {', '.join(sorted(medidas))} — "
     f"{len(medidas)} de {len(todas)}.  \n"
     f"**Heredadas:** {', '.join(heredadas)} — "
     f"{len(heredadas)} de {len(todas)}, fuera de muestra.")

## 5 · El sello


In [ ]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())